# CloudStore - File Storage System Simulation

This notebook simulates the CloudStore architecture:

`Client -> API Gateway -> Metadata Service -> Chunk Service -> Block Service -> Storage Nodes`
`                              |`
`                       Users & Devices Service`

Each service is a separate class with its **own SQLite database** (database-per-service principle).
Storage Nodes are simulated as local folders with **replication** for fault tolerance.

**Run the cells in order (top to bottom).** Setup cells (1-7) must run first since later task cells depend on them. After that, each numbered task cell in Section 8 can be re-run individually.

## 1. Imports & Configuration

In [ ]:
import os
import sqlite3
import hashlib
import shutil
import uuid
import time

# Chunk size: 1 MB chunks (real systems use 4-8MB; small here for demo)
CHUNK_SIZE = 1024 * 1024

# Each chunk is stored on 3 different nodes for durability/fault tolerance
REPLICATION_FACTOR = 3

# Simulates the "Cloud blob store" (S3-equivalent)
STORAGE_ROOT = "cloudstore_data"

# Simulates a distributed storage cluster
NUM_STORAGE_NODES = 5

# Separate databases per service - mirrors "database separation per service"
METADATA_DB = "metadata_service.db"
USERS_DEVICES_DB = "users_devices.db"
CHUNK_DB = "chunk_service.db"

print("Configuration loaded.")

## 2. Storage Cluster (Block Service + Cloud Blob Store)

Simulates a cluster of distributed storage nodes. Each chunk is replicated across `REPLICATION_FACTOR` nodes using a consistent-hashing-style placement, so the system tolerates node failures (Q6: Scalability & Fault Tolerance).

In [ ]:
class StorageCluster:
    def __init__(self, root=STORAGE_ROOT, num_nodes=NUM_STORAGE_NODES):
        self.root = root
        self.num_nodes = num_nodes
        self.node_dirs = []
        for i in range(num_nodes):
            node_path = os.path.join(root, f"node_{i}")
            os.makedirs(node_path, exist_ok=True)
            self.node_dirs.append(node_path)
        self.failed_nodes = set()

    def _pick_nodes(self, chunk_hash, count):
        """Consistent-hashing style: deterministic but spread-out node placement."""
        start = int(chunk_hash, 16) % self.num_nodes
        return [(start + i) % self.num_nodes for i in range(count)]

    def store_chunk(self, chunk_hash, data):
        nodes = self._pick_nodes(chunk_hash, REPLICATION_FACTOR)
        written_to = []
        for node_id in nodes:
            if node_id in self.failed_nodes:
                continue
            path = os.path.join(self.node_dirs[node_id], chunk_hash)
            with open(path, "wb") as f:
                f.write(data)
            written_to.append(node_id)
        if not written_to:
            raise RuntimeError(f"Chunk {chunk_hash[:8]} could not be stored - all target nodes are down!")
        return written_to

    def read_chunk(self, chunk_hash):
        nodes = self._pick_nodes(chunk_hash, REPLICATION_FACTOR)
        for node_id in nodes:
            if node_id in self.failed_nodes:
                continue
            path = os.path.join(self.node_dirs[node_id], chunk_hash)
            if os.path.exists(path):
                with open(path, "rb") as f:
                    return f.read()
        raise FileNotFoundError(f"Chunk {chunk_hash[:8]} unavailable - all its replicas are missing or on failed nodes.")

    def simulate_node_failure(self, node_id):
        print(f"  [FAULT TOLERANCE] Storage node {node_id} has gone DOWN.")
        self.failed_nodes.add(node_id)

    def recover_node(self, node_id):
        print(f"  [FAULT TOLERANCE] Storage node {node_id} is back UP.")
        self.failed_nodes.discard(node_id)

print("StorageCluster defined.")

## 3. Metadata Service

Owns file metadata: name, owner, size, version, folder path, and permissions. Backed by its own SQLite database (`metadata_service.db`).

In [ ]:
class MetadataService:
    def __init__(self, db_path=METADATA_DB):
        self.conn = sqlite3.connect(db_path)
        self._init_schema()

    def _init_schema(self):
        cur = self.conn.cursor()
        cur.execute("""
            CREATE TABLE IF NOT EXISTS files (
                file_id TEXT PRIMARY KEY,
                file_name TEXT NOT NULL,
                owner_id TEXT NOT NULL,
                folder_path TEXT DEFAULT '/',
                size_bytes INTEGER,
                latest_version INTEGER DEFAULT 1,
                created_at REAL,
                updated_at REAL
            )
        """)
        cur.execute("""
            CREATE TABLE IF NOT EXISTS permissions (
                file_id TEXT,
                user_id TEXT,
                access_level TEXT,
                PRIMARY KEY (file_id, user_id)
            )
        """)
        self.conn.commit()

    def create_file(self, file_name, owner_id, size_bytes, folder_path="/"):
        file_id = str(uuid.uuid4())
        now = time.time()
        cur = self.conn.cursor()
        cur.execute("""
            INSERT INTO files (file_id, file_name, owner_id, folder_path, size_bytes, latest_version, created_at, updated_at)
            VALUES (?, ?, ?, ?, ?, 1, ?, ?)
        """, (file_id, file_name, owner_id, folder_path, size_bytes, now, now))
        cur.execute("INSERT INTO permissions (file_id, user_id, access_level) VALUES (?, ?, 'owner')", (file_id, owner_id))
        self.conn.commit()
        return file_id

    def bump_version(self, file_id, new_size_bytes):
        cur = self.conn.cursor()
        cur.execute("UPDATE files SET latest_version = latest_version + 1, size_bytes = ?, updated_at = ? WHERE file_id = ?",
                     (new_size_bytes, time.time(), file_id))
        self.conn.commit()
        return self.get_file(file_id)["latest_version"]

    def get_file(self, file_id):
        cur = self.conn.cursor()
        cur.execute("SELECT file_id, file_name, owner_id, folder_path, size_bytes, latest_version, created_at, updated_at FROM files WHERE file_id = ?", (file_id,))
        row = cur.fetchone()
        if not row:
            return None
        cols = ["file_id", "file_name", "owner_id", "folder_path", "size_bytes", "latest_version", "created_at", "updated_at"]
        return dict(zip(cols, row))

    def share_file(self, file_id, user_id, access_level="read"):
        cur = self.conn.cursor()
        cur.execute("INSERT OR REPLACE INTO permissions (file_id, user_id, access_level) VALUES (?, ?, ?)", (file_id, user_id, access_level))
        self.conn.commit()

    def check_access(self, file_id, user_id, required="read"):
        cur = self.conn.cursor()
        cur.execute("SELECT access_level FROM permissions WHERE file_id = ? AND user_id = ?", (file_id, user_id))
        row = cur.fetchone()
        if not row:
            return False
        levels = {"read": 1, "write": 2, "owner": 3}
        return levels.get(row[0], 0) >= levels.get(required, 999)

    def list_files(self, user_id):
        cur = self.conn.cursor()
        cur.execute("""
            SELECT f.file_id, f.file_name, f.size_bytes, f.latest_version, p.access_level
            FROM files f JOIN permissions p ON f.file_id = p.file_id
            WHERE p.user_id = ?
        """, (user_id,))
        return cur.fetchall()

print("MetadataService defined.")

## 4. Chunk Service

Splits files into chunks, hashes each chunk (SHA-256) for deduplication and integrity, tracks chunk order per file version, and hands chunks off to the `StorageCluster`.

In [ ]:
class ChunkService:
    def __init__(self, storage, db_path=CHUNK_DB):
        self.storage = storage
        self.conn = sqlite3.connect(db_path)
        self._init_schema()

    def _init_schema(self):
        cur = self.conn.cursor()
        cur.execute("""
            CREATE TABLE IF NOT EXISTS file_chunks (
                file_id TEXT,
                version INTEGER,
                chunk_index INTEGER,
                chunk_hash TEXT,
                chunk_size INTEGER,
                PRIMARY KEY (file_id, version, chunk_index)
            )
        """)
        self.conn.commit()

    @staticmethod
    def _hash_chunk(data):
        return hashlib.sha256(data).hexdigest()

    def split_into_chunks(self, data):
        return [data[i:i + CHUNK_SIZE] for i in range(0, len(data), CHUNK_SIZE)]

    def upload_file_content(self, file_id, version, data):
        chunks = self.split_into_chunks(data)
        cur = self.conn.cursor()
        for idx, chunk in enumerate(chunks):
            chunk_hash = self._hash_chunk(chunk)
            cur.execute("SELECT 1 FROM file_chunks WHERE chunk_hash = ? LIMIT 1", (chunk_hash,))
            already_exists = cur.fetchone() is not None
            if not already_exists:
                nodes = self.storage.store_chunk(chunk_hash, chunk)
                print(f"    Chunk {idx} (hash {chunk_hash[:8]}...) -> stored on nodes {nodes}")
            else:
                print(f"    Chunk {idx} (hash {chunk_hash[:8]}...) -> deduplicated (already exists)")
            cur.execute("""
                INSERT OR REPLACE INTO file_chunks (file_id, version, chunk_index, chunk_hash, chunk_size)
                VALUES (?, ?, ?, ?, ?)
            """, (file_id, version, idx, chunk_hash, len(chunk)))
        self.conn.commit()
        return len(chunks)

    def download_file_content(self, file_id, version):
        cur = self.conn.cursor()
        cur.execute("SELECT chunk_index, chunk_hash FROM file_chunks WHERE file_id = ? AND version = ? ORDER BY chunk_index ASC", (file_id, version))
        rows = cur.fetchall()
        if not rows:
            raise FileNotFoundError("No chunks found for this file/version")
        data = b""
        for idx, chunk_hash in rows:
            data += self.storage.read_chunk(chunk_hash)
        return data

print("ChunkService defined.")

## 5. Users & Devices Service

Manages users, storage quota, and registered devices. Backed by its own database - separate from file metadata, demonstrating the "database separation per service" design principle.

In [ ]:
class UserDeviceService:
    DEFAULT_QUOTA_BYTES = 5 * 1024 * 1024 * 1024  # 5 GB

    def __init__(self, db_path=USERS_DEVICES_DB):
        self.conn = sqlite3.connect(db_path)
        self._init_schema()

    def _init_schema(self):
        cur = self.conn.cursor()
        cur.execute("""
            CREATE TABLE IF NOT EXISTS users (
                user_id TEXT PRIMARY KEY,
                name TEXT,
                quota_bytes INTEGER,
                used_bytes INTEGER DEFAULT 0
            )
        """)
        cur.execute("""
            CREATE TABLE IF NOT EXISTS devices (
                device_id TEXT PRIMARY KEY,
                user_id TEXT,
                device_name TEXT,
                last_sync REAL
            )
        """)
        self.conn.commit()

    def register_user(self, user_id, name, quota_bytes=None):
        cur = self.conn.cursor()
        cur.execute("INSERT OR IGNORE INTO users (user_id, name, quota_bytes, used_bytes) VALUES (?, ?, ?, 0)",
                     (user_id, name, quota_bytes or self.DEFAULT_QUOTA_BYTES))
        self.conn.commit()

    def register_device(self, user_id, device_name):
        device_id = str(uuid.uuid4())
        cur = self.conn.cursor()
        cur.execute("INSERT INTO devices (device_id, user_id, device_name, last_sync) VALUES (?, ?, ?, ?)",
                     (device_id, user_id, device_name, time.time()))
        self.conn.commit()
        return device_id

    def has_quota(self, user_id, additional_bytes):
        cur = self.conn.cursor()
        cur.execute("SELECT quota_bytes, used_bytes FROM users WHERE user_id = ?", (user_id,))
        quota, used = cur.fetchone()
        return (used + additional_bytes) <= quota

    def consume_quota(self, user_id, bytes_used):
        cur = self.conn.cursor()
        cur.execute("UPDATE users SET used_bytes = used_bytes + ? WHERE user_id = ?", (bytes_used, user_id))
        self.conn.commit()

    def get_usage(self, user_id):
        cur = self.conn.cursor()
        cur.execute("SELECT quota_bytes, used_bytes FROM users WHERE user_id = ?", (user_id,))
        return cur.fetchone()

print("UserDeviceService defined.")

## 6. Notification Service

Simulates the Notification Queue + Notification Server. Online devices get notified immediately; offline devices get their notification stored in a backlog (simulating the Redis in-memory backlog) until they reconnect.

In [ ]:
class NotificationService:
    def __init__(self):
        self.queue = []      # simulates Notification Queue
        self.backlog = {}    # device_id -> list of pending notifications

    def publish_event(self, file_id, file_name, event_type, target_devices):
        event = {"file_id": file_id, "file_name": file_name, "event": event_type, "timestamp": time.time()}
        self.queue.append(event)
        for device_id, is_online in target_devices.items():
            if is_online:
                print(f"    [NOTIFY] Device {device_id[:8]} -> '{event_type}' for {file_name}")
            else:
                self.backlog.setdefault(device_id, []).append(event)
                print(f"    [NOTIFY] Device {device_id[:8]} is offline - event queued in backlog")

    def reconnect_device(self, device_id):
        pending = self.backlog.pop(device_id, [])
        if pending:
            print(f"    [NOTIFY] Device {device_id[:8]} reconnected - delivering {len(pending)} backlog event(s)")
        else:
            print(f"    [NOTIFY] Device {device_id[:8]} reconnected - no backlog events")
        return pending

print("NotificationService defined.")

## 7. CloudStore Facade (API Gateway)

High-level facade that mimics the API Gateway routing requests to the correct backend service. This is the class client code interacts with.

In [ ]:
class CloudStore:
    def __init__(self):
        self.storage = StorageCluster()
        self.metadata = MetadataService()
        self.chunks = ChunkService(self.storage)
        self.users = UserDeviceService()
        self.notifications = NotificationService()

    def upload_file(self, owner_id, file_name, data, target_devices=None):
        size = len(data)
        if not self.users.has_quota(owner_id, size):
            raise PermissionError("Upload rejected: storage quota exceeded.")
        file_id = self.metadata.create_file(file_name, owner_id, size)
        print(f"  [METADATA] Created file '{file_name}' (id={file_id[:8]})")
        num_chunks = self.chunks.upload_file_content(file_id, version=1, data=data)
        print(f"  [CHUNK] Split into {num_chunks} chunk(s), version 1")
        self.users.consume_quota(owner_id, size)
        if target_devices:
            self.notifications.publish_event(file_id, file_name, "file_uploaded", target_devices)
        return file_id

    def update_file(self, owner_id, file_id, new_data, target_devices=None):
        if not self.metadata.check_access(file_id, owner_id, required="write"):
            raise PermissionError("User does not have write access to this file.")
        old_size = self.metadata.get_file(file_id)["size_bytes"]
        new_size = len(new_data)
        delta = new_size - old_size
        if delta > 0 and not self.users.has_quota(owner_id, delta):
            raise PermissionError("Update rejected: storage quota exceeded.")
        new_version = self.metadata.bump_version(file_id, new_size)
        self.chunks.upload_file_content(file_id, version=new_version, data=new_data)
        self.users.consume_quota(owner_id, max(delta, 0))
        print(f"  [METADATA] File {file_id[:8]} updated to version {new_version}")
        if target_devices:
            file_name = self.metadata.get_file(file_id)["file_name"]
            self.notifications.publish_event(file_id, file_name, "file_updated", target_devices)
        return new_version

    def download_file(self, requester_id, file_id, version=None):
        if not self.metadata.check_access(file_id, requester_id, required="read"):
            raise PermissionError("Access denied: no read permission for this file.")
        meta = self.metadata.get_file(file_id)
        version = version or meta["latest_version"]
        data = self.chunks.download_file_content(file_id, version)
        print(f"  [DOWNLOAD] Retrieved '{meta['file_name']}' v{version} ({len(data)} bytes)")
        return data

    def share_file(self, file_id, with_user_id, access_level="read"):
        self.metadata.share_file(file_id, with_user_id, access_level)
        print(f"  [METADATA] Shared file {file_id[:8]} with {with_user_id} ({access_level} access)")

print("CloudStore facade defined.")

---
# 8. Demo Tasks (run individually)

Run **Task 0** once to reset everything and create a fresh `CloudStore` instance (`cs`). After that, **Tasks 1-9 can each be run independently** to demonstrate a specific feature/requirement.

### Task 0: Setup / Reset
Cleans up any previous run's database files and storage folders, then creates a fresh `CloudStore` instance and two test users with devices. **Run this first.**

In [ ]:
# Cleanup previous run
for f in [METADATA_DB, USERS_DEVICES_DB, CHUNK_DB]:
    if os.path.exists(f):
        os.remove(f)
if os.path.exists(STORAGE_ROOT):
    shutil.rmtree(STORAGE_ROOT)

cs = CloudStore()

cs.users.register_user("alice", "Alice")
cs.users.register_user("bob", "Bob")
alice_phone = cs.users.register_device("alice", "Alice's Phone")
alice_laptop = cs.users.register_device("alice", "Alice's Laptop")
bob_desktop = cs.users.register_device("bob", "Bob's Desktop")

print("Setup complete.")
print(f"Alice's devices: phone={alice_phone[:8]}, laptop={alice_laptop[:8]}")
print(f"Bob's device: desktop={bob_desktop[:8]}")

### Task 1: Chunked File Upload (Q3 - File Upload)
Alice uploads a ~2.5MB file. It gets split into 1MB chunks, each hashed and stored across replicated storage nodes. Alice's phone is online (notified instantly); her laptop is offline (notification goes to backlog).

In [ ]:
fake_video = os.urandom(int(2.5 * CHUNK_SIZE))  # ~2.5 chunks of random bytes simulating a video file

target_devices = {alice_phone: True, alice_laptop: False}  # laptop offline
file_id = cs.upload_file("alice", "trip_video.mp4", fake_video, target_devices)

print(f"\nfile_id = {file_id}")

### Task 2: Offline Device Reconnects (Notification Backlog)
Alice's laptop comes back online and receives the notification that was queued in the backlog while it was offline.

In [ ]:
backlog_events = cs.notifications.reconnect_device(alice_laptop)
print("\nBacklog events delivered:", backlog_events)

### Task 3: File Sharing & Access Control (Q1/Q4 - Permissions)
Alice shares the video with Bob, granting read-only access.

In [ ]:
cs.share_file(file_id, "bob", access_level="read")

### Task 4: Chunked Download & Integrity Check (Q3 - File Retrieval)
Bob downloads the shared file. The Chunk Service reassembles it from chunks retrieved across storage nodes, and we verify the result matches the original byte-for-byte.

In [ ]:
downloaded = cs.download_file("bob", file_id)
print(f"\nIntegrity check: downloaded == original -> {downloaded == fake_video}")

### Task 5: File Versioning & Deduplication (Q3/Q4 - Versioning)
Alice uploads a new version where the first chunk is unchanged and the second is new. The Chunk Service detects the unchanged chunk's hash already exists and **deduplicates** it instead of storing it again.

In [ ]:
updated_video = fake_video[:CHUNK_SIZE] + os.urandom(CHUNK_SIZE)  # 1st chunk reused -> dedup
new_version = cs.update_file("alice", file_id, updated_video,
                              target_devices={alice_phone: True, alice_laptop: True})
print(f"\nNew version number: {new_version}")

### Task 6: Permission Enforcement (Q1 - Security)
Bob (read-only access) attempts to overwrite the file. The system should reject this with a `PermissionError`.

In [ ]:
try:
    cs.update_file("bob", file_id, b"hacked content")
except PermissionError as e:
    print(f"Blocked as expected: {e}")

### Task 7: Fault Tolerance - Storage Node Failure (Q6)
Two of the five storage nodes go down. Because each chunk is replicated across `REPLICATION_FACTOR=3` nodes, the file should still be downloadable successfully.

In [ ]:
cs.storage.simulate_node_failure(0)
cs.storage.simulate_node_failure(1)

try:
    data = cs.download_file("alice", file_id)
    print(f"\nFile still downloaded successfully despite 2 node failures "
          f"({len(data)} bytes) - thanks to replication factor {REPLICATION_FACTOR}")
except FileNotFoundError as e:
    print(f"Download failed: {e}")

# Recover nodes for subsequent tasks
cs.storage.recover_node(0)
cs.storage.recover_node(1)

### Task 8: Storage Quota Check (Q1 - Non-functional requirement)
Display Alice's current quota usage, and demonstrate that an upload exceeding her remaining quota is rejected (without actually generating gigabytes of data).

In [ ]:
usage = cs.users.get_usage("alice")
print(f"Alice's usage: {usage[1]} / {usage[0]} bytes ({usage[1] / usage[0] * 100:.4f}% used)")

remaining = usage[0] - usage[1]
oversized_request = remaining + 1
if not cs.users.has_quota("alice", oversized_request):
    print(f"\nBlocked as expected: upload of {oversized_request} bytes "
          f"would exceed remaining quota of {remaining} bytes.")

### Task 9: Inspect Stored Data (Database & Storage Layout)
Lists Alice's files from the Metadata Service, the chunk records from the Chunk Service, and shows how chunks are physically distributed across the simulated storage nodes.

In [ ]:
print("Alice's files (Metadata Service):")
for row in cs.metadata.list_files("alice"):
    print("  ", row)

print("\nChunk records (Chunk Service):")
cur = cs.chunks.conn.cursor()
cur.execute("SELECT file_id, version, chunk_index, chunk_hash, chunk_size FROM file_chunks ORDER BY version, chunk_index")
for row in cur.fetchall():
    print("  ", row)

print("\nStorage node contents (Block Service):")
for node_dir in cs.storage.node_dirs:
    files = os.listdir(node_dir)
    print(f"  {node_dir}: {len(files)} chunk(s) -> {files}")